# Data overview

Two kinds of investor document, both **synthetic or public, never real client data**:

| Document | Source | Why |
|---|---|---|
| Passport, ID card | [DocXPand-25k](https://github.com/QuickSign/docxpand): fictional IDs (state "Galathée", generated faces) photographed and scanned on real backgrounds | Looks like real uploads, and every field has an answer key |
| Subscription agreement | Real **blank** Reg A forms from SEC EDGAR (4 issuers), filled with the same synthetic person | Real layouts, several of them, so nothing is tuned to one form |

Each subscription agreement arrives one of three ways: **digital** (filled PDF with a text layer),
**scanned** (image-only PDF), or **photo** (phone picture). Some files carry planted issues.

In [ ]:
from pathlib import Path
import os, sys
if Path(os.getcwd()).name == "notebooks":
    os.chdir("..")  # run from the repo root so the pipeline imports work
sys.path.insert(0, os.getcwd())
import json
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from config import DEMO

labels = json.loads((DEMO / "labels.json").read_text())
df = pd.DataFrame([{"investor": l["investor_id"], "id": l["id_kind"], "form": l["form"],
                    "condition": l["condition"], "scenario": l["scenario"]} for l in labels])
df.head()

## What's in the demo set

In [ ]:
pd.concat([df["id"].value_counts(), df["condition"].value_counts(), df["scenario"].value_counts()],
          keys=["ID type", "Subscription arrived as", "Planted scenario"]).to_frame("investors")

## One investor's upload

File names are neutral (`upload_1.jpg`...): the pipeline has to work out which document is which.

In [ ]:
def show_upload(investor_id, width=5):
    folder = DEMO / investor_id
    files = sorted(p for p in folder.iterdir() if p.is_file())
    fig, axes = plt.subplots(1, len(files), figsize=(width * len(files), width * 1.2))
    for ax, path in zip(axes, files):
        from extract.pages import page_images
        ax.imshow(page_images(path)[0])
        ax.set_title(path.name)
        ax.axis("off")
    plt.tight_layout()

# An investor whose subscription was photographed (an ID card holder if there is one)
photos = df[df["condition"] == "photo"].sort_values("id")
show_upload(photos["investor"].iloc[0])

## The same subscription form, three ways it can arrive

In [ ]:
for condition in ["digital", "scanned", "photo"]:
    print(condition, "→", df[df["condition"] == condition]["investor"].iloc[0])